# Image Classification - DSAI Lab Week 3
**Strategy**: EfficientNetV2-S transfer learning with fine-tuning, aggressive data augmentation, and Test-Time Augmentation (TTA)

This notebook trains a state-of-the-art image classifier for 6 scene categories:
`buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`

In [2]:
# ── 1. IMPORTS & SETUP ────────────────────────────────────────────────────────
import os
import random
import glob as _glob
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, applications
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow version:', tf.__version__)
print('GPUs available:', len(gpus))
print('Devices:', tf.config.list_logical_devices())


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ── 2. CONFIGURATION & AUTO-DETECT PATHS ─────────────────────────────────────
import glob as _glob

# ── Show all available inputs for diagnosis ───────────────────────────────────
_all_inputs = sorted(_glob.glob('/kaggle/input/*'))
print('All available inputs under /kaggle/input/:')
for _p in _all_inputs:
    print('  ', _p)

if not _all_inputs:
    raise RuntimeError(
        '\n\nNo dataset found under /kaggle/input/!\n'
        'FIX: In the Kaggle notebook editor:\n'
        '  1. Look at the right sidebar → "Input" section\n'
        '  2. Click the arrow (▶) next to the competition name\n'
        '  3. Click the "+" or "Add" button to attach it as input\n'
        '  4. Re-run this cell.\n'
    )

# ── Auto-detect BASE_DIR: pick the input folder that has training class folders ─
BASE_DIR = None
for _candidate in _all_inputs:
    _candidate_path = Path(_candidate)
    # Check if this folder (or any subfolder) contains the 'buildings' class
    _hits = list(_candidate_path.rglob('buildings'))
    if _hits:
        BASE_DIR = _candidate_path
        break

if BASE_DIR is None:
    # Fallback: just use the first available input
    BASE_DIR = Path(_all_inputs[0])
    print(f'WARNING: Could not find "buildings" folder. Using: {BASE_DIR}')
else:
    print(f'\nDataset root selected: {BASE_DIR}')

# ── Auto-detect TRAIN_DIR ─────────────────────────────────────────────────────
_buildings_matches = sorted(BASE_DIR.rglob('buildings'))
if not _buildings_matches:
    raise RuntimeError(
        f'"buildings" class folder not found inside {BASE_DIR}.\n'
        'Make sure you added the correct competition dataset as input.'
    )
TRAIN_DIR = _buildings_matches[0].parent
print('TRAIN_DIR resolved:', TRAIN_DIR)

# ── Auto-detect TEST_DIR ──────────────────────────────────────────────────────
_img0_matches = sorted(BASE_DIR.rglob('img000.jpg'))
if not _img0_matches:
    raise RuntimeError(
        '"img000.jpg" not found. Check that the competition test set is included in the dataset.'
    )
TEST_DIR = _img0_matches[0].parent
print('TEST_DIR resolved :', TEST_DIR)

# ── Sample submission ─────────────────────────────────────────────────────────
_sample_matches = sorted(BASE_DIR.rglob('sample_submission.csv'))
if not _sample_matches:
    raise RuntimeError('sample_submission.csv not found in dataset.')
SAMPLE = _sample_matches[0]
print('SAMPLE CSV        :', SAMPLE)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE   = 224   # EfficientNetV2-S native size
BATCH_SIZE = 32
EPOCHS_FT  = 12   # Phase 1: frozen backbone
EPOCHS_UF  = 18   # Phase 2: fine-tune top layers
LR_FT      = 3e-4
LR_UF      = 5e-5
UNFREEZE_N = 100  # last N backbone layers to unfreeze
TTA_STEPS  = 8    # Test-Time Augmentation rounds
DROPOUT    = 0.40

CLASSES     = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES = len(CLASSES)
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
print('\nClasses:', CLASSES)
print('\n✓ All paths resolved successfully!')


In [ ]:
# ── 3. BUILD TRAINING DATAFRAME ───────────────────────────────────────────────
records = []
for cls in CLASSES:
    cls_dir = TRAIN_DIR / cls
    if not cls_dir.exists():
        print(f'WARNING: class folder missing: {cls_dir}')
        continue
    imgs = (list(cls_dir.glob('*.jpg')) +
            list(cls_dir.glob('*.jpeg')) +
            list(cls_dir.glob('*.png')))
    for img_path in imgs:
        records.append({'path': str(img_path), 'label': cls})

if not records:
    print('ERROR: No images found! Contents of TRAIN_DIR:')
    for p in sorted(TRAIN_DIR.iterdir()):
        print('  ', p)
    raise RuntimeError('No training images found. Check TRAIN_DIR above.')

train_df = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Total training images: {len(train_df)}')
print(train_df['label'].value_counts())


In [ ]:
# ── 4. CLASS WEIGHTS (handle imbalance) ───────────────────────────────────────
labels_arr = train_df['label'].map(CLASS2IDX).values
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=labels_arr)
class_weight_dict = dict(enumerate(weights))
print('Class weights:', class_weight_dict)

In [ ]:
# ── 5. DATASET PIPELINE ───────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# Augmentation layers (Keras built-in, runs on GPU)
train_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomTranslation(0.10, 0.10),
], name='train_augmentation')

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label

def augment(img, label):
    # Keras aug layers handle both 3D (H,W,C) and 4D (B,H,W,C) inputs
    img = train_aug(img, training=True)
    return img, label

def load_test_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img

# 80/20 train/val split (both reset index for safety)
n_val     = int(len(train_df) * 0.20)
val_df    = train_df[:n_val].reset_index(drop=True)
train_df2 = train_df[n_val:].reset_index(drop=True)

def make_train_ds(df):
    paths  = df['path'].values
    labels = df['label'].map(CLASS2IDX).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(len(df), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.map(augment,    num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

def make_val_ds(df):
    paths  = df['path'].values
    labels = df['label'].map(CLASS2IDX).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_train_ds(train_df2)
val_ds   = make_val_ds(val_df)
print(f'Train batches: {len(train_ds)}, Val batches: {len(val_ds)}')
print(f'Train samples: {len(train_df2)}, Val samples: {len(val_df)}')


In [ ]:

# ── 6. MODEL: MobileNetV2 + Custom Head (weights pre-cached in Kaggle) ────────
# MobileNetV2 ImageNet weights are bundled in Kaggle's TF Docker image —
# no internet download needed, no Kaggle Model input needed.

def build_model(trainable_base=False, unfreeze_n=0):
    """Build MobileNetV2 with custom classification head."""
    base = applications.MobileNetV2(
        include_top=False,
        weights='imagenet',          # pre-cached on disk in Kaggle env
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    if not trainable_base:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:-unfreeze_n]:
            layer.trainable = False
        for layer in base.layers[-unfreeze_n:]:
            layer.trainable = not isinstance(layer, layers.BatchNormalization)

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    # MobileNetV2 expects inputs scaled to [-1, 1]
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT / 2)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return keras.Model(inputs, outputs)

model = build_model(trainable_base=False)
model.summary(line_length=100)
print('\n✓ Model built successfully.')


In [ ]:
# ── 7. PHASE 1 TRAINING: Frozen backbone ─────────────────────────────────────
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_FT),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cbs_phase1 = [
    callbacks.ModelCheckpoint(
        '/kaggle/working/best_phase1.keras',
        monitor='val_accuracy', mode='max',
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    )
]

print('=== PHASE 1: Training head (frozen backbone) ===')
hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    class_weight=class_weight_dict,
    callbacks=cbs_phase1,
    verbose=1
)


In [ ]:
# ── 8. PHASE 2 TRAINING: Fine-tune top backbone layers ────────────────────────
model_ft = build_model(trainable_base=True, unfreeze_n=UNFREEZE_N)

# Transfer best weights from Phase 1 into the new model
model_ft.set_weights(model.get_weights())

# CosineDecayRestarts as optimizer LR schedule — smooth warm restarts during fine-tuning.
# NOTE: Do NOT add ReduceLROnPlateau here — it conflicts with a LearningRateSchedule
# (optimizer.lr is no longer a plain tf.Variable when a schedule is used).
cosine_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LR_UF,
    first_decay_steps=len(train_ds) * 4,
    t_mul=2.0,
    m_mul=0.85
)

model_ft.compile(
    optimizer=optimizers.Adam(learning_rate=cosine_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cbs_phase2 = [
    callbacks.ModelCheckpoint(
        '/kaggle/working/best_model.keras',
        monitor='val_accuracy', mode='max',
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7,
        restore_best_weights=True, verbose=1
    ),
    # No ReduceLROnPlateau here — cosine schedule handles LR decay
]

print('=== PHASE 2: Fine-tuning top backbone layers ===')
hist2 = model_ft.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_UF,
    class_weight=class_weight_dict,
    callbacks=cbs_phase2,
    verbose=1
)


In [ ]:

# ── 9. LOAD BEST MODEL & EVALUATE ────────────────────────────────────────────
import os as _os

# Use best Phase 2 model if it exists, otherwise fall back to Phase 1
_model_path = '/kaggle/working/best_model.keras'
_fallback   = '/kaggle/working/best_phase1.keras'

if _os.path.exists(_model_path):
    print(f'Loading Phase 2 model: {_model_path}')
    best_model = keras.models.load_model(_model_path)
elif _os.path.exists(_fallback):
    print(f'Phase 2 model not found — loading Phase 1 fallback: {_fallback}')
    best_model = keras.models.load_model(_fallback)
else:
    # Last resort: use the in-memory model_ft (or model) from training
    print('WARNING: No saved checkpoint found — using in-memory model from last training phase.')
    try:
        best_model = model_ft
    except NameError:
        best_model = model

# Evaluate on validation set
val_preds_raw = best_model.predict(val_ds, verbose=1)
val_preds     = np.argmax(val_preds_raw, axis=1)
val_true      = val_df['label'].map(CLASS2IDX).values

wf1 = f1_score(val_true, val_preds, average='weighted')
print(f'\nValidation Weighted F1-Score: {wf1:.4f}')
print('\nClassification Report:')
print(classification_report(val_true, val_preds, target_names=CLASSES))


In [ ]:

# ── 10. TRAINING CURVES ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

try:
    all_acc   = hist1.history['accuracy'] + hist2.history['accuracy']
    all_val   = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
    all_loss  = hist1.history['loss'] + hist2.history['loss']
    all_vloss = hist1.history['val_loss'] + hist2.history['val_loss']
    phase1_end = len(hist1.history['accuracy']) - 1
except NameError:
    # hist2 may not exist if Phase 2 was skipped or errored
    all_acc   = hist1.history['accuracy']
    all_val   = hist1.history['val_accuracy']
    all_loss  = hist1.history['loss']
    all_vloss = hist1.history['val_loss']
    phase1_end = None

axes[0].plot(all_acc, label='Train Acc')
axes[0].plot(all_val, label='Val Acc')
if phase1_end:
    axes[0].axvline(phase1_end, color='r', ls='--', label='Phase 2 start')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(all_loss, label='Train Loss')
axes[1].plot(all_vloss, label='Val Loss')
if phase1_end:
    axes[1].axvline(phase1_end, color='r', ls='--', label='Phase 2 start')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True)

plt.suptitle(f'Training Curves  |  Val Weighted F1 = {wf1:.4f}')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()


In [ ]:
# ── 11. TEST-TIME AUGMENTATION (TTA) PREDICTION ───────────────────────────────
sample_df = pd.read_csv(SAMPLE)
test_ids   = sample_df['id'].tolist()   # e.g. ['img000', 'img001', ...]
test_paths = [str(TEST_DIR / f'{tid}.jpg') for tid in test_ids]

print(f'Test images: {len(test_paths)}')

# Build TTA augmentation (mild, inference-time)
tta_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
], name='tta_augmentation')

def make_test_ds(paths, augment_fn=None):
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(load_test_image, num_parallel_calls=AUTOTUNE)
    if augment_fn is not None:
        ds = ds.map(lambda x: augment_fn(x, training=True), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

print('Running TTA inference...')

# Original (no augmentation)
base_ds = make_test_ds(test_paths)
tta_probs = best_model.predict(base_ds, verbose=0)

# TTA rounds with augmentation
for t in range(TTA_STEPS):
    tta_ds = make_test_ds(test_paths, augment_fn=tta_aug)
    tta_probs += best_model.predict(tta_ds, verbose=0)
    print(f'  TTA round {t+1}/{TTA_STEPS} done')

tta_probs /= (TTA_STEPS + 1)  # average over all rounds
test_preds = np.argmax(tta_probs, axis=1)
test_labels = [CLASSES[p] for p in test_preds]

print('\nPrediction distribution:')
pd.Series(test_labels).value_counts()

In [ ]:
# ── 12. CREATE SUBMISSION FILE ────────────────────────────────────────────────
submission = pd.DataFrame({
    'id':     test_ids,
    'target': test_labels
})

# Verify ordering matches sample submission exactly
assert list(submission['id']) == list(sample_df['id']), 'ID order mismatch!'

# Save to Kaggle working directory (this is the output Kaggle reads for submission)
OUTPUT_PATH = '/kaggle/working/submission.csv'
submission.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(submission.head(10))
print(f'\nTotal predictions: {submission.shape[0]}')
print('\nClass distribution:')
print(submission['target'].value_counts())


In [ ]:
# ── 13. SAMPLE PREDICTIONS VISUALIZATION ─────────────────────────────────────
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
sample_indices = np.random.choice(len(test_paths), 18, replace=False)

for ax, idx in zip(axes.flatten(), sample_indices):
    img = Image.open(test_paths[idx]).resize((224, 224))
    confidence = tta_probs[idx].max()
    pred_class = test_labels[idx]
    ax.imshow(img)
    ax.set_title(f'{test_ids[idx]}\n{pred_class} ({confidence:.2f})', fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Test Predictions (with confidence)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=100)
plt.show()

print('\n✓ All done! Submit submission.csv to Kaggle.')